In [1]:
import os
import time
import zipfile
import json
import gc
from pathlib import Path
from typing import Dict, List, Tuple, Any
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# =========================================================================
# GLOBAL RUNTIME CONFIGURATIONS
# =========================================================================
DATA_DIR = Path("/content/olchiki_digits") if IN_COLAB else Path("./olchiki_digits")
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

IMAGE_SIZE = 64        # Expanded to 64x64 to preserve precise Ol Chiki geometric intersections
BATCH_SIZE = 128       # Maximizes parallel GPU tensor generation pipelines
EPOCHS = 35
LEARNING_RATE = 0.002
RANDOM_STATE = 42

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Ol Chiki Core Target Locked: {device}")

def upload_and_extract_zip() -> Path:
    if DATA_DIR.exists() and any(DATA_DIR.iterdir()):
        print(f"⚡ [SKIPPED] Active workspace detected at '{DATA_DIR}'. Bypassing extraction.")
        return DATA_DIR

    zip_path = None
    content_zips = list(Path("/content").glob("*.zip")) if IN_COLAB else list(Path(".").glob("*.zip"))
    if content_zips:
        zip_path = sorted(content_zips, key=lambda x: x.stat().st_mtime)[-1]
    else:
        if IN_COLAB:
            uploaded = files.upload()
            if not uploaded: raise RuntimeError("X Action terminated.")
            zip_path = Path("/content") / next(iter(uploaded.keys()))
        else:
            raise FileNotFoundError("X Source zip data package missing.")

    print(f" Unpacking character data archive: {zip_path}...")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(DATA_DIR)
    return DATA_DIR

# =========================================================================
# GEOMETRY-PRESERVING GLYPH LOADER ENGINE
# =========================================================================
class OlChikiDigitsDataset(Dataset):
    def __init__(self, folder_path: Path, class_names: List[str], class_to_idx: Dict[str, int], is_train: bool = False):
        self.samples = []
        self.is_train = is_train

        print(f" Mapping and index checking directories in: {folder_path.name}...")
        for class_name in class_names:
            class_folder = folder_path / class_name
            if not class_folder.exists(): continue
            idx = class_to_idx[class_name]
            for file_path in class_folder.iterdir():
                if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
                    if file_path.stat().st_size > 0:
                        self.samples.append((file_path, idx))

    def letterbox_pad(self, img: Image.Image, target_size: int) -> Image.Image:
        """Pads the image to a square format to prevent script skewing."""
        w, h = img.size
        max_dim = max(w, h)
        # Create a solid black background canvas (inverted logic)
        canvas = Image.new('L', (max_dim, max_dim), 0)
        offset = ((max_dim - w) // 2, (max_dim - h) // 2)
        canvas.paste(img, offset)
        return canvas.resize((target_size, target_size), Image.Resampling.BILINEAR)

    def __len__(self): return len(self.samples)

    def __getitem__(self, item):
        file_path, label = self.samples[item]
        try:
            with Image.open(file_path) as img:
                # Strip out RGBA Alpha transparency channels
                if img.mode == 'RGBA':
                    canvas = Image.new('RGBA', img.size, (255, 255, 255))
                    canvas.paste(img, (0, 0), img)
                    img = canvas.convert('L')
                else:
                    img = img.convert('L')

                # Invert logic: guarantees high active values on characters and dampens backgrounds
                img = ImageOps.invert(img)

                if self.is_train:
                    if np.random.rand() > 0.40:
                        angle = np.random.uniform(-12, 12) # Slight rotation bounds
                        img = img.rotate(angle, resample=Image.Resampling.BILINEAR)
                    if np.random.rand() > 0.50:
                        # Controlled shearing to emulate handwriting variations
                        shear_x = np.random.uniform(-0.15, 0.15)
                        img = img.transform(img.size, Image.Transform.AFFINE,
                                            (1, shear_x, 0, 0, 1, 0), resample=Image.Resampling.BILINEAR)

                img = self.letterbox_pad(img, IMAGE_SIZE)
                arr = np.array(img, dtype=np.float32) / 255.0
                tensor = torch.tensor(arr, dtype=torch.float32).unsqueeze(0)
                return (tensor - 0.5) / 0.5, label
        except Exception:
            return torch.zeros((1, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.float32), label

# =========================================================================
# EXTREME PARAMETER-EFFICIENT HARDWARE LAYERS
# =========================================================================
class MetaMish(nn.Module):
    def __init__(self):
        super().__init__()
        self.beta = nn.Parameter(torch.tensor(1.0))
    def forward(self, x):
        return x * torch.tanh(F.softplus(x * self.beta))

class CoordConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels + 2, out_channels, **kwargs)
    def forward(self, x):
        b, _, h, w = x.size()
        y_coord = torch.linspace(-1, 1, h, device=x.device).view(1, 1, h, 1).expand(b, 1, h, w)
        x_coord = torch.linspace(-1, 1, w, device=x.device).view(1, 1, 1, w).expand(b, 1, h, w)
        return self.conv(torch.cat([x, y_coord, x_coord], dim=1))

class DenseBlockLayer(nn.Module):
    def __init__(self, in_channels: int, growth_rate: int):
        super().__init__()
        self.bn = nn.BatchNorm2d(in_channels)
        self.act = MetaMish()
        self.conv = nn.Conv2d(in_channels, growth_rate, kernel_size=3, padding=1, bias=False)
    def forward(self, x):
        out = self.conv(self.act(self.bn(x)))
        return torch.cat([x, out], dim=1)

# =========================================================================
# AUTOREGRESSIVE COMPACT CASCADE NETWORK
# =========================================================================
class Stage1DenseFeatureExtractor(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        # CoordConv helps track the absolute positioning of stroke vertices
        self.stem = nn.Sequential(
            CoordConv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            MetaMish()
        )

        # Interconnected structural routing blocks
        self.layer1 = DenseBlockLayer(64, 32)   # 96 channels
        self.layer2 = DenseBlockLayer(96, 32)   # 128 channels
        self.layer3 = DenseBlockLayer(128, 64)  # 192 channels

        self.transition = nn.Sequential(
            nn.Conv2d(192, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128),
            MetaMish(),
            nn.MaxPool2d(2, 2)  # Output scale space: 32x32
        )

        self.layer4 = DenseBlockLayer(128, 64)  # 192 channels
        self.layer5 = DenseBlockLayer(192, 64)  # 256 channels

        self.pool_classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer3(self.layer2(self.layer1(x)))
        spatial_chain = self.transition(x)
        spatial_chain = self.layer5(self.layer4(spatial_chain))
        logits = self.pool_classifier(spatial_chain)
        return logits, spatial_chain

class Stage2SpatialRefiner(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.compressor = nn.Sequential(
            nn.Conv2d(256, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            MetaMish()
        )
        self.processing = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),  # 16x16
            nn.BatchNorm2d(128), MetaMish(),
            nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=1), # 8x8
            nn.BatchNorm2d(128), MetaMish()
        )
        self.raw_ingestion = nn.Sequential(
            CoordConv2d(1, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), MetaMish(),
            nn.MaxPool2d(4, 4),                                     # 16x16
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # 8x8
            nn.BatchNorm2d(64), MetaMish()
        )
        self.multi_scale_pool = nn.Sequential(
            nn.Conv2d(128 + 64, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            MetaMish(),
            nn.AdaptiveAvgPool2d((2, 2))  # Retains global structural quadrant markers
        )

        self.highway_1 = nn.Sequential(nn.Linear(1024 + num_classes, 4200), nn.BatchNorm1d(4200), MetaMish(), nn.Dropout(0.25))
        self.highway_2 = nn.Sequential(nn.Linear(4200, 4200), nn.BatchNorm1d(4200), MetaMish(), nn.Dropout(0.25))
        self.head = nn.Linear(4200, num_classes)

    def forward(self, x, stage1_logits, spatial_chain):
        compressed = self.compressor(spatial_chain)
        feats_chain = self.processing(compressed)
        feats_raw = self.raw_ingestion(x)

        fused = torch.cat([feats_chain, feats_raw], dim=1)
        spatial_map = self.multi_scale_pool(fused).view(fused.size(0), -1)

        meta_vector = torch.cat([spatial_map, stage1_logits], dim=1)
        h1 = self.highway_1(meta_vector)
        h2 = self.highway_2(h1) + h1
        return self.head(h2)

class CascadedOlChikiNetwork(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.stage1 = Stage1DenseFeatureExtractor(num_classes)
        self.stage2 = Stage2SpatialRefiner(num_classes)
    def forward(self, x):
        s1_logits, spatial_chain = self.stage1(x)
        s2_logits = self.stage2(x, s1_logits, spatial_chain)
        return s2_logits, s1_logits

# =========================================================================
# REPORT COMPILATION HUB
# =========================================================================
def generate_html_report(history: Dict[str, List[float]], metrics: Dict[str, Any], total_params: int) -> str:
    history_json = json.dumps(history)
    report_cleaned = "<br>".join(metrics["class_report"].split("\n"))
    accuracy = metrics["accuracy"] * 100
    return f"""<!DOCTYPE html>
    <html lang="en" class="dark">
    <head>
        <meta charset="UTF-8"><title>Ol Chiki Digits Analytics Dashboard</title>
        <script src="https://cdn.tailwindcss.com"></script>
        <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
        <style>body {{ background: #030712; color: #f3f4f6; font-family: sans-serif; }}</style>
    </head>
    <body class="p-12">
        <header class="max-w-7xl mx-auto mb-10 flex justify-between border-b border-gray-800 pb-6">
            <div>
                <h1 class="text-3xl font-extrabold text-white">Ol Chiki Digits Cascaded Optimization Framework</h1>
                <p class="text-sm text-amber-400 mt-1">Aspect-Preserved Multi-Scale Spatial Coordinate Feature Mapping</p>
            </div>
            <div class="bg-gray-900 px-8 py-3 rounded-2xl text-center border border-amber-500/40">
                <p class="text-xs uppercase text-gray-400 tracking-wider font-semibold font-mono">Validation Accuracy</p>
                <span class="text-4xl font-black text-emerald-400 font-mono">{accuracy:.2f}%</span>
            </div>
        </header>
        <main class="max-w-7xl mx-auto space-y-8">
            <div class="bg-gray-900 p-6 rounded-2xl border border-emerald-500/20">
                <p class="text-emerald-400 font-mono text-base">⚙️ Model Scale: {total_params:,} Trainable System Parameters Calibrated</p>
            </div>
            <div class="grid grid-cols-1 lg:grid-cols-2 gap-8">
                <div class="bg-gray-900 p-6 rounded-2xl border border-gray-800"><canvas id="lossChart" height="220"></canvas></div>
                <div class="bg-gray-900 p-6 rounded-2xl border border-gray-800"><canvas id="accuracyChart" height="220"></canvas></div>
            </div>
            <div class="bg-gray-900 p-8 rounded-2xl border border-gray-800">
                <h3 class="text-md font-bold text-amber-400 mb-4 font-mono">Precision / Recall Sub-Class Performance Summary Logs</h3>
                <pre class="text-xs text-gray-300 font-mono overflow-x-auto p-4 bg-black/40 rounded-xl">{report_cleaned}</pre>
            </div>
        </main>
        <script>
            const data = {history_json};
            new Chart(document.getElementById('lossChart'), {{ type: 'line', data: {{ labels: Array.from({{length: data.loss.length}}, (_,i)=>i+1), datasets: [{{label:'Optimization Matrix Loss', data:data.loss, borderColor:'#f59e0b', fill:false, tension:0.1}}] }} }});
            new Chart(document.getElementById('accuracyChart'), {{ type: 'line', data: {{ labels: Array.from({{length: data.acc.length}}, (_,i)=>i+1), datasets: [{{label:'Training Accuracy Path', data:data.acc.map(a=>a*100), borderColor:'#10b981', fill:false, tension:0.1}}] }} }});
        </script>
    </body></html>"""

# =========================================================================
# RUNTIME CONTROLLER
# =========================================================================
def main():
    print("=========================================================")
    print("  COMMENCING HIGH-PRECISION OL CHIKI RUNTIME ENGINE     ")
    print("=========================================================")

    data_path = upload_and_extract_zip()

    train_dir, test_dir = data_path / "train", data_path / "test"
    if not train_dir.exists():
        for subdir in data_path.rglob("train"):
            if subdir.is_dir():
                train_dir = subdir
                test_dir = subdir.parent / "test"
                break

    class_names = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    num_classes = len(class_names)
    print(f" Identified Ol Chiki Sub-classes: {num_classes} -> {class_names}")

    train_dataset = OlChikiDigitsDataset(train_dir, class_names, class_to_idx, is_train=True)
    test_dataset = OlChikiDigitsDataset(test_dir, class_names, class_to_idx, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    pipeline = CascadedOlChikiNetwork(num_classes=num_classes).to(device)
    total_params = sum(p.numel() for p in pipeline.parameters() if p.requires_grad)
    print(f" Calculated Structural Memory Blueprint: {total_params:,} Trainable System Parameters.")

    criterion = nn.CrossEntropyLoss(label_smoothing=0.04)
    optimizer = optim.AdamW(pipeline.parameters(), lr=LEARNING_RATE, weight_decay=0.02)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')

    history = {"loss": [], "acc": []}

    print(f"\n Processing optimizations across {EPOCHS} structured hardware evaluation loops...\n")
    for epoch in range(EPOCHS):
        start = time.time()
        pipeline.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda' if device.type == 'cuda' else 'cpu'):
                s2_logits, s1_logits = pipeline(images)
                loss_s1 = criterion(s1_logits, labels)
                loss_s2 = criterion(s2_logits, labels)
                combined_loss = (0.30 * loss_s1) + loss_s2

            scaler.scale(combined_loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss_s2.item() * images.size(0)
            _, predicted = torch.max(s2_logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        scheduler.step()
        epoch_loss = running_loss / max(1, len(train_dataset))
        epoch_acc = correct / max(1, total)
        history["loss"].append(epoch_loss)
        history["acc"].append(epoch_acc)

        print(f"Cycle [{epoch+1}/{EPOCHS}] ({time.time()-start:.1f}s) -> Composite Loss: {epoch_loss:.4f} | System Accuracy: {epoch_acc*100:.2f}%")

    # Evaluation phase
    pipeline.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            s2_logits, _ = pipeline(images)
            _, predicted = torch.max(s2_logits, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_preds, all_labels = np.array(all_preds), np.array(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    p, r, _, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
    class_rep = classification_report(all_labels, all_preds, target_names=class_names, zero_division=0)

    metrics = {"accuracy": float(acc), "precision_weighted": float(p), "recall_weighted": float(r), "class_report": class_rep}
    print(f"\n🎯 Training Completed. Verified Target Score Matrix: {acc*100:.2f}%")

    html_report = generate_html_report(history, metrics, total_params)
    report_filename = "olchiki_digits_optimized_report.html"
    with open(report_filename, "w", encoding="utf-8") as f: f.write(html_report)
    if IN_COLAB: files.download(report_filename)
    print(" Execution successfully finalized.")

if __name__ == "__main__":
    main()

🚀 Ol Chiki Core Target Locked: cuda
  COMMENCING HIGH-PRECISION OL CHIKI RUNTIME ENGINE     


Saving olchiki_digits_dataset.zip to olchiki_digits_dataset.zip
 Unpacking character data archive: /content/olchiki_digits_dataset.zip...
 Identified Ol Chiki Sub-classes: 10 -> ['digit_0', 'digit_1', 'digit_2', 'digit_3', 'digit_4', 'digit_5', 'digit_6', 'digit_7', 'digit_8', 'digit_9']
 Mapping and index checking directories in: train...
 Mapping and index checking directories in: test...
 Calculated Structural Memory Blueprint: 23,085,875 Trainable System Parameters.

 Processing optimizations across 35 structured hardware evaluation loops...

Cycle [1/35] (22.7s) -> Composite Loss: 1.7406 | System Accuracy: 70.56%
Cycle [2/35] (20.2s) -> Composite Loss: 0.4921 | System Accuracy: 93.56%
Cycle [3/35] (20.3s) -> Composite Loss: 0.4202 | System Accuracy: 95.69%
Cycle [4/35] (20.5s) -> Composite Loss: 0.3963 | System Accuracy: 96.12%
Cycle [5/35] (20.7s) -> Composite Loss: 0.3221 | System Accuracy: 98.38%
Cycle [6/35] (21.0s) -> Composite Loss: 0.3017 | System Accuracy: 98.86%
Cycle [7/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Execution successfully finalized.
